In [0]:
dbutils.widgets.text("source_type", "csv", "source_type")
dbutils.widgets.text("path", "/Workspace/Users/sateeshfrnd@gmail.com/SAMBA/data_to_model.csv", "path")

# /Workspace/Users/sateeshfrnd@gmail.com/SAMBA/datareadiness_assesment/sample_datasets/data_to_model.csv

In [0]:
def validate_source_type(source_type):
    SUPPORTED_SOURCE_TYPES = ["unity_catalog","deltalake","datalake","msks","csv", "excel",]
    if source_type not in SUPPORTED_SOURCE_TYPES:
        raise Exception(f"Source type {source_type} is not supported. Supported source types are {SUPPORTED_SOURCE_TYPES}")
    return source_type


In [0]:
source_type = validate_source_type(dbutils.widgets.get("source_type"))
source_type

In [0]:
source_type = dbutils.widgets.get("source_type")
source_type

In [0]:


file_path = dbutils.widgets.get("path")

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)
display(df)

In [0]:
# ==========================================================
# User Story : US03
# Title      : Identify & Review Data Quality Issues
# Purpose    : Perform Data Quality Assessment and Capability Evaluation
# Author     : <Your Name>
# ==========================================================

# =====================================================================================
# User Story ID : US03
# User Story    : Identify & Review Data Quality Issues
#
# Objective:
# Validate dataset quality and assess readiness for analytics and AI/ML workloads.
#
# Data Quality Checks:
# 1. Duplicate Rows
# 2. Empty Rows
# 3. Empty Columns
# 4. Duplicate Columns
# 5. Null Values Validation
# 6. Data Type Validation
# 7. Missing Data Validation
# 8. Constant / Near Constant Feature Validation
#
# Capability Checks:
# 1. Data Intake
# 2. Predictor Quality
# 3. Missingness & Completeness
# 4. Sample Size & Coverage
# 5. Structural Integrity
# 6. Final Recommendation
#
# Output:
# - DQ Summary Report
# - Capability Assessment Report
# - Data Readiness Recommendation (GO / NO-GO)
# =====================================================================================

# Scenario 1: Issue Detection

Given a dataset has been loaded When data quality assessment is executed Then the system shall identify:
- Duplicate rows
- Duplicate columns
- Empty rows
- Empty columns
- Missing values
- Invalid data types
- Schema inconsistencies
- Constant columns
- Near-constant columns

In [0]:
dq_results = []

##1. Duplicate Rows


In [0]:
total_rows = df.count() # total count
distinct_rows = df.distinct().count() # Unique records
duplicate_rows = total_rows - distinct_rows
dq_results.append(
("Duplicate Rows",
duplicate_rows,
"True" if duplicate_rows > 0 else "False")
)
print(dq_results)

## 2. Duplicate columns

In [0]:
from collections import Counter

# Check duplicate column names
column_counts = Counter(df.columns)

duplicate_columns = [
    col_name
    for col_name, cnt in column_counts.items()
    if cnt > 1
]

# Append result to DQ report
dq_results.append(
    (
        "Duplicate Columns",
        len(duplicate_columns),
        "True" if len(duplicate_columns) > 0 else "False"
    )
)


# Display Results
display(
    spark.createDataFrame(
        dq_results,
        ["Issue", "Count", "Status"]
    )
)

## Empty rows

In [0]:
#2. Empty rows
from pyspark.sql.functions import *
condition = None
for c in df.columns:
    expr = col(c).isNull() | (trim(col(c).cast("string")) == "")
    if condition is None:
       condition = expr
else:
    condition = condition & expr
empty_rows = df.filter(condition).count()
dq_results.append(
(
    "Empty Rows",
     empty_rows,
    "True" if empty_rows > 0 else "False"
)
)
display(
spark.createDataFrame(
dq_results,
["Issue", "Count", "Severity"]
)
)
print(dq_results)

## Empty columns

In [0]:

#3.Empty Columns

from pyspark.sql.functions import *
from pyspark.sql import functions as F


empty_columns = []

for c in df.columns:

    non_empty_count = (
        df.filter(
            F.col(c).isNotNull() &
            (F.trim(F.col(c).cast("string")) != "")
        ).count()
    )

    if non_empty_count == 0:
        empty_columns.append(c)

dq_results.append(
    (
        "Empty Columns",
        len(empty_columns),
        "True" if len(empty_columns) > 0 else "False"
    )
)

# ==================================================
# Display Results
# ==================================================
display(
    spark.createDataFrame(
        dq_results,
        ["Issue", "Count", "Status"]
    )
)

print(dq_results)

## Missing values

In [0]:
# ==================================================
# Scenario 5: Null Values
# ==================================================

from pyspark.sql import functions as F

results = []
total_nulls = 0

for c in df.columns:

    null_cnt = df.filter(F.col(c).isNull()).count()

    na_cnt = df.filter(
        F.lower(F.trim(F.col(c).cast("string"))) == "na"
    ).count()

    n_a_cnt = df.filter(
        F.lower(F.trim(F.col(c).cast("string"))) == "n/a"
    ).count()

    nan_cnt = df.filter(
        F.lower(F.trim(F.col(c).cast("string"))) == "nan"
    ).count()

    blank_cnt = df.filter(
        F.trim(F.col(c).cast("string")) == ""
    ).count()

    # Total issues for the column
    column_total = (
        null_cnt +
        na_cnt +
        n_a_cnt +
        nan_cnt +
        blank_cnt
    )

    total_nulls += column_total

    results.append(
        (
            c,
            null_cnt,
            na_cnt,
            n_a_cnt,
            nan_cnt,
            blank_cnt
        )
    )

# Detailed Report
display(
    spark.createDataFrame(
        results,
        ["Column_Name", "NULL", "NA", "N/A", "NaN", "Blank"]
    )
)

# DQ Summary
dq_results.append(
    (
        "Null Value Check",
        total_nulls,
        "True" if total_nulls > 0 else "False"
    )
)

# Display DQ Results
display(
    spark.createDataFrame(
        dq_results,
        ["Issue", "Count", "Status"]
    )
)

## Invalid data types

In [0]:
expected_schema= {
    "_c0": "string",
    "UID": "string",
    "RootRecipeCode": "string",
    "Electrolyte": "double",
    "Fats and oils": "double",
    "Fatty acid": "double",
    "Filler": "double",
    "Free alkali": "double",
    "Free fatty acid": "double",
    "Humectant": "double",
    "Minor": "double",
    "Perfume": "double",
    "Polymer": "double",
    "Soap noodles": "double",
    "Structurant": "double",
    "Surfactant": "double",
    "Water": "double",
    "Electrolytes": "double",
    "FattyMatter": "double",
    "Other Filler": "double",
    "Starch": "double",
    "Talc": "double",
    "Mean_Iodine_value": "double",
    "C8": "double",
    "C10": "double",
    "C12": "double",
    "C14": "double",
    "C16": "double",
    "C18": "double",
    "C18_1": "double",
    "C18_2": "double",
    "C18_3": "double",
    "C20": "double",
    "other": "double",
    "ShortChainSat": "double",
    "LongChain": "double",
    "LongChainUnsat": "double",
    "LongChainSat": "double",
    "RatioC12C14": "double",
    "AveragedAbsorbedWater": "double",
    "BarHardnessDuringUse": "integer",
    "BarSmoothDryFeel": "integer",
    "BarStickinessBeforeUse": "integer",
    "BubbleSize": "integer",
    "ColourOfFoam": "integer",
    "Grittiness25C": "integer",
    "Grittiness40C": "integer",
    "LatherConsistency": "integer",
    "LatherQuantity": "integer",
    "LatherSpeedGeneration": "integer",
    "MilkyLather": "double",
    "AverageLossOfMass": "double",
    "AverageMushAmount": "double",
    "MushCurdledaspect": "integer",
    "MushDryaspect": "integer",
    "MushHardnessFeel": "integer",
    "MushJellyAspect": "integer",
    "MushPastyaspect": "integer",
    "MushQuantityFeeling": "integer",
    "MushQuantityVisual": "integer",
    "MushSlimyViscousAspect": "integer",
    "MushStickiness": "integer",
    "ResidueonSoapDish": "integer",
    "Sandiness25C": "integer",
    "Sandiness40C": "integer",
    "Slipperiness25C": "integer",
    "Slipperiness40C": "integer",
    "SwollenBar": "integer",
    "WhitePatches": "integer",
    "WhiteSlurryorDyeonSoapDish": "double",
    "AverageWearRate": "double",
    "Rinse": "double",
    "LatherVolume": "double",
    "Total_Cracking_Score": "integer",
    "Total_Cracking_Score_days": "integer",
    "RateofWearMean": "double"
  }


In [0]:
#Scenario 4: Data Type Validation
# Read schema from config file
# expected_schema = config["expected_schema"]

actual_schema = dict(df.dtypes)

validation_results = []
mismatched_columns = []

for col_name, expected_dtype in expected_schema.items():

    actual_dtype = actual_schema.get(
        col_name,
        "Column Missing"
    )

    is_match = actual_dtype == expected_dtype

    if not is_match:
        mismatched_columns.append(col_name)

    validation_results.append(
        (
            col_name,
            expected_dtype,
            actual_dtype,
            is_match,
            "Matched" if is_match else "Mismatched"
        )
    )

schema_df = spark.createDataFrame(
    validation_results,
    [
        "Column_Name",
        "Expected_DataType",
        "Actual_DataType",
        "Boolean_Check",
        "Status"
    ]
)
dq_results.append(
(
"Data Type Validation",
len(mismatched_columns),
"True" if len(mismatched_columns) > 0 else "False"
)
)
display(
    spark.createDataFrame(
        dq_results,
        ["Issue", "Count", "Status"]
    )
)
display(schema_df)

In [0]:
matched_df = schema_df.filter("Boolean_Check = true")

display(matched_df)

In [0]:
mismatched_df = schema_df.filter("Boolean_Check = false")

display(mismatched_df)

## Schema inconsistencies

## Constant columns

## Near-constant columns


# Scenario 4: Data Type Validation

Given a dataset has been loaded
When quality assessment is executed
Then the system shall identify columns containing invalid or inconsistent data types

# Scenario 5: Missing Data Validation

Given a dataset has been loaded
When missingness analysis is executed
Then the system shall identify columns exceeding configured missing value thresholds

In [0]:
dq_rules= {
    "missing_threshold": 10,
    "near_constant_threshold": 95,
    "hard_stop_threshold": 50
  }

In [0]:
#Scenario 5: Missing Data Validation
from pyspark.sql import functions as F

# Configure missing value threshold (%)
# missing_threshold = config["dq_rules"]["missing_threshold"]
missing_threshold = dq_rules["missing_threshold"]

total_rows = df.count()

results = []

for column_name in df.columns:

    missing_count = df.filter(
        F.col(column_name).isNull() |
        (F.trim(F.col(column_name).cast("string")) == "")
    ).count()

    missing_percentage = ((missing_count * 100.0) / total_rows)

    results.append(
        (
            column_name,
            missing_count,
            missing_percentage,
            "Threshold Exceeded" if missing_percentage > missing_threshold else "Within Threshold"
        )
    )

missing_validation_df = spark.createDataFrame(
    results,
    [
        "Column_Name",
        "Missing_Count",
        "Missing_Percentage",
        "Validation_Status"
    ]
)

display(missing_validation_df)

print("Columns Exceeding Missing Value Threshold:")

display(
    missing_validation_df.filter(
        F.col("Validation_Status") == "Threshold Exceeded"
    )
)
threshold_exceeded_count = missing_validation_df.filter(
    F.col("Validation_Status") == "Threshold Exceeded"
).count()

dq_results.append(
(
    "Missing Data Validation",
    threshold_exceeded_count,
    "True" if threshold_exceeded_count > 0 else "False"
)
)

display(
    spark.createDataFrame(
        dq_results,
        ["Issue", "Count", "Status"]
    )
)

# Scenario 6: Constant Feature Validation

Given a dataset contains columns with little or no variation
When quality assessment is executed
Then the system shall identify constant and near-constant columns


In [0]:
#Scenario 6: Constant Feature Validation
from pyspark.sql import functions as F
constant_columns = []
near_constant_columns = []

total_rows = df.count()

for c in df.columns:

    # Constant Column Check
    distinct_count = df.select(F.countDistinct(F.col(c))).first()[0]

    if distinct_count == 1:
        constant_columns.append(c)

    # Near Constant Column Check (95% threshold)
    top_count = (
        df.groupBy(c)
           .count()
           .orderBy(F.desc("count"))
           .first()
    )

    if top_count is not None:

        dominant_count = top_count["count"]
        dominant_percentage = (dominant_count * 100.0) / total_rows

        if dominant_percentage >= 95 and distinct_count > 1:
            near_constant_columns.append(c)

# Add to DQ Summary
dq_results.append(
    (
        "Constant Columns",
        len(constant_columns),
        "True" if len(constant_columns) > 0 else "False"
    )
)

dq_results.append(
    (
        "Near Constant Columns",
        len(near_constant_columns),
        "True" if len(near_constant_columns) > 0 else "False"
    )
)

print("Constant Columns Found:")
for col_name in constant_columns:
    print(col_name)

print("\nNear Constant Columns Found:")
for col_name in near_constant_columns:
    print(col_name)

# DQ Summary Report
display(
    spark.createDataFrame(
        dq_results,
        ["Issue", "Count", "Status"]
    )
)


# display columns

#feature_results = []

#for c in constant_columns:
 #   feature_results.append((c, "Constant"))

#for c in near_constant_columns:
 #   feature_results.append((c, "Near Constant"))

#feature_df = spark.createDataFrame(
 #   feature_results,
  #  ["Column_Name", "Feature_Type"]
#)

#display(feature_df)




In [0]:
feature_results = []

for c in constant_columns:
    feature_results.append((c, "Constant"))

for c in near_constant_columns:
    feature_results.append((c, "Near Constant"))

feature_df = spark.createDataFrame(
    feature_results,
    ["Column_Name", "Feature_Type"]
)

display(feature_df)